In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import numbers

from pandas.core.dtypes.common import is_numeric_dtype

### Dataset Source
Link: https://www.kaggle.com/datasets/adilshamim8/economic-indicators-and-inflation/

In [ ]:
# Load Datasets
economic_indicators_and_inflation = pd.read_csv('../data/task03/Economic Indicators And Inflation.csv')
development_indicators = pd.read_excel('../data/task03/WorldBank.xlsx')

print(f'Economic Indicators and Inflation:\n{economic_indicators_and_inflation}')
print('----------------------------------')
print(f'Development Indicators:\n{development_indicators}')

In [ ]:
# Functions

def headers_to_snake_case(df):
    """Renames headers to snake_case"""
    df = df.copy()

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip('_')
    )
    return df


def check_values_below_zero(df):
    """Checks if there are any values below zero"""
    for col_name in df.columns:
        if is_numeric_dtype(df[col_name]):
            invalid_value = df.loc[
                (df[col_name] < 0)
            ]
            print(f'Below zero values in "{col_name}": {len(invalid_value)}')

    return f'All Columns checked'


def add_categories(
        df,
        col,
        category_col_name,
        bins,
        labels,
        right=True,
        retbins=False,
        precision=3,
        include_lowest=False,
        duplicates='raise',
        ordered=True
):
    """Creates categories with pd.cut()"""
    df[category_col_name] = pd.cut(
        df[col],
        bins=bins,
        labels=labels,
        right=right,
        retbins=retbins,
        precision=precision,
        include_lowest=include_lowest,
        duplicates=duplicates,
        ordered=ordered
    )
    return df


def get_q1_q3(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    return q1, q3


def highest(df, by_col):
    """Gets the highest value"""
    try:
        return df.loc[df[by_col].idxmax()]
    except KeyError:
        print(f"No column named {by_col}")
        return None


def lowest(df, by_col):
    """Gets the lowest value"""
    try:
        return df.loc[df[by_col].idxmin()]
    except KeyError:
        print(f"No column named {by_col}")


def format_value(value):
    """Format value if it is a number"""
    if isinstance(value, numbers.Number):
        return f"{value:.2f}"
    else:
        return str(value)


def highest_print(rank_type, name, rank_value):
    return (f'Highest average {rank_type.lower()}:'
            f'\n- {name}'
            f'\n- {rank_type}: {format_value(rank_value)}')


def lowest_print(rank_type, name, rank_value):
    return (f'Lowest average {rank_type.lower()}:'
            f'\n- {name}'
            f'\n- {rank_type}: {format_value(rank_value)}')


def first_and_last_year(df, group_by, col, sort_values, first_col, last_col):
    """
    Creates a table with first year, last year and the values from them.
    Helps calculate change over time
    """
    new_df = (
        df
        .dropna(subset=[col])
        .sort_values(sort_values)
        .groupby(group_by)
        .agg(
            first_year=('year', 'first'),
            last_year=('year', 'last'),
            first_col=(col, 'first'),
            last_col=(col, 'last'),
        )
        .reset_index()
    )

    new_df = new_df.rename(columns={
        'first_col': first_col,
        'last_col': last_col,
        }
    )

    return new_df


def calculate_change(df, first, last, col_name,percent=True):
    """Calculates change over time"""
    df[f'{col_name}_change_percent'] = (
        df[last] - df[first]
    )

    if not percent:
        df = df.rename(columns={f'{col_name}_change_percent': f'{col_name}_change'})
        df[f'{col_name}_change_percent'] = (
            df[f'{col_name}_change'] / df[first] * 100
        )

    return df

def top3(df, by_col, ascending=True):
    """Identifies the top 3 by a certain value"""
    df =  df.sort_values(by=by_col, ascending=ascending).head(3)

    return df


In [ ]:
# Shape
print(f'Economic Indicators and Inflation: {economic_indicators_and_inflation.shape}')
print(f'Development Indicators: {development_indicators.shape}')

In [ ]:
# Column Headers

print(f'Economic Indicators and Inflation Headers: {', '.join(economic_indicators_and_inflation.columns)}')
print()
print(f'Development Indicators Headers: {', '.join(development_indicators.columns)}')

### Possible Similar columns
Economic Indicators and Inflation  - Development Indicators
- Country and Country Name
- Year and Year
- Unemployment Rate (%) and Unemployment (% of total labor force) (modeled ILO estimate)

###### ***This will be analyzed further before merging the tables***

#### Inspect Datasets

In [ ]:
# Economic Indicators and Inflation Dataset Info
economic_indicators_and_inflation.info()

In [ ]:
# Economic Indicators and Inflation Dataset Header Titles
economic_indicators_and_inflation.columns

In [ ]:
# Rename Headers of Economic Indicators and Inflation
economic_indicators_and_inflation = headers_to_snake_case(economic_indicators_and_inflation)
economic_indicators_and_inflation.columns

In [ ]:
# Checking for Null Values in Economic Indicators and Inflation Dataset
economic_indicators_and_inflation.isna().sum()

In [ ]:
# Development Indicators Dataset Info
development_indicators.info()

In [ ]:
# Development Indicators Dataset Header Titles
development_indicators.columns

In [ ]:
# Rename Development Indicators Headers
development_indicators = headers_to_snake_case(development_indicators)
development_indicators.columns

In [ ]:
development_indicators = development_indicators.rename(columns={'incomegroup': 'income_group'})
development_indicators.columns

In [ ]:
# Checking for Null Values in Development Indicators
development_indicators.isna().sum().head(20)

In [ ]:
print(
    f'Economic Indicators and Inflation \n '
    f'min year: {economic_indicators_and_inflation.year.min()}\n '
    f'max year: {economic_indicators_and_inflation.year.max()}'
)
print()
print(
    f'Development Indicators \n '
    f'min year: {development_indicators.year.min()}\n '
    f'max year: {development_indicators.year.max()}'
)

#### The years in common are from 2010 to 2018
I will reduce both datasets to the years in common

In [ ]:
econom_ind_and_inflation_2010_2018 = economic_indicators_and_inflation.loc[
    economic_indicators_and_inflation['year'].between(2010, 2018)
]

dev_ind_2010_2018 = development_indicators.loc[
    development_indicators['year'].between(2010, 2018)
]

print(econom_ind_and_inflation_2010_2018)
print()
print(dev_ind_2010_2018)

In [ ]:
# NaN values in Development Indicators after reducing dataframe
dev_ind_2010_2018.isna().sum()

The World Bank development indicators contain missing values because not all countries report all indicators for every year. Since the dataset contains many indicators, removing all rows with missing values would result in unnecessary data loss. Therefore, missing values will be  handled selectively. Rows will be not removed globally. Instead, missing values will be kept as NaN, and calculations will be performed using available observations. For analyses requiring specific indicators, only rows with missing values in those specific columns will be excluded. Columns with very high missingness, such as electric power consumption, are kept but will be not used as core indicators in the main analysis.

In [ ]:
# Check percentage of missing values
missing_summary = (
    dev_ind_2010_2018
    .isna()
    .sum()
    .to_frame('missing_count')
)

missing_summary['missing_percent'] = (
    missing_summary['missing_count'] / len(dev_ind_2010_2018) * 100
)

print(missing_summary.sort_values('missing_percent', ascending=False))

In [ ]:
# Important and optional columns
core_dev_ind_cols = [
    "country_name",
    "country_code",
    "region",
    "income_group",
    "year",
    "gdp_usd",
    "gdp_per_capita_usd",
    "individuals_using_the_internet_of_population",
    "life_expectancy_at_birth_years",
    "population_density_people_per_sq_km_of_land_area",
    "unemployment_of_total_labor_force_modeled_ilo_estimate",
    "birth_rate_crude_per_1_000_people",
    "death_rate_crude_per_1_000_people"
]

optional_dev_ind_cols = [
    "electric_power_consumption_kwh_per_capita",
    "infant_mortality_rate_per_1_000_live_births"
]

In [ ]:
# Development Indicators core table
dev_ind_2010_2018_core = dev_ind_2010_2018[core_dev_ind_cols].copy()
dev_ind_2010_2018_core

In [ ]:
# Duplicate rows
print(
    f'Economic Indicators and Inflation duplicated rows: '
      f'{econom_ind_and_inflation_2010_2018.duplicated().sum()}'
)

print(f'Development Indicators duplicated rows: {dev_ind_2010_2018_core.duplicated().sum()}')


In [ ]:
econom_ind_and_inflation_2010_2018.info()

In [ ]:
# Values bellow zero
dataframes = [econom_ind_and_inflation_2010_2018, dev_ind_2010_2018_core]
for df in dataframes:
    print(check_values_below_zero(df))

Two columns contain values below zero: inflation_rate and economic_growth. Both are part of the Economic Indicators and Inflation dataset. Negative inflation rates are economically meaningful because they represent deflation, while negative economic growth values indicate economic contraction. Therefore, these values were not removed during the cleaning process.

In [ ]:
# Potential outliers
economic_indicators_and_inflation_outliers = econom_ind_and_inflation_2010_2018.loc[
    (econom_ind_and_inflation_2010_2018['inflation_rate'] < -10) |
    (econom_ind_and_inflation_2010_2018['inflation_rate'] > 20)  |
    (econom_ind_and_inflation_2010_2018['economic_growth'] < -10) |
    (econom_ind_and_inflation_2010_2018['economic_growth'] > 20)
]

economic_indicators_and_inflation_outliers

In [ ]:
dev_ind_outliers = dev_ind_2010_2018_core.loc[
    (dev_ind_2010_2018_core["individuals_using_the_internet_of_population"] < 0) |
    (dev_ind_2010_2018_core["individuals_using_the_internet_of_population"] > 100) |

    (dev_ind_2010_2018_core["life_expectancy_at_birth_years"] < 40) |
    (dev_ind_2010_2018_core["life_expectancy_at_birth_years"] > 90) |

    (dev_ind_2010_2018_core["population_density_people_per_sq_km_of_land_area"] < 0) |
    (dev_ind_2010_2018_core["population_density_people_per_sq_km_of_land_area"] > 10000) |

    (dev_ind_2010_2018_core["unemployment_of_total_labor_force_modeled_ilo_estimate"] < 0) |
    (dev_ind_2010_2018_core["unemployment_of_total_labor_force_modeled_ilo_estimate"] > 100) |

    (dev_ind_2010_2018_core["birth_rate_crude_per_1_000_people"] < 0) |
    (dev_ind_2010_2018_core["birth_rate_crude_per_1_000_people"] > 60) |

    (dev_ind_2010_2018_core["death_rate_crude_per_1_000_people"] < 0) |
    (dev_ind_2010_2018_core["death_rate_crude_per_1_000_people"] > 30)
]

dev_ind_outliers

Both datasets contain potential outliers. In the Economic Indicators and Inflation dataset, potential outliers were identified in the inflation_rate and economic_growth columns. In the Development Indicators dataset, potential outliers were identified in the population_density_people_per_sq_km_of_land_area column. These values were not removed automatically, because extreme values may be valid for specific countries and years. Instead, they were flagged for further review.

In [ ]:
# Convert GDP from a billion USD to USD in the Economic Indicators dataset
econom_ind_and_inflation_2010_2018['gdb_usd_economic'] = (
    econom_ind_and_inflation_2010_2018['gdp_in_billion_usd'] * 1_000_000_000
)

econom_ind_and_inflation_2010_2018

In [ ]:
# Rename Development Indicator GDP column for clarity when merging both tables in the future
dev_ind_2010_2018_core = dev_ind_2010_2018_core.rename(
    columns={
        'gdp_usd': 'gdb_usd_worldbank',
    }
)

dev_ind_2010_2018_core

#### Merge Datasets

In [ ]:
# Merge check
merge_check = pd.merge(
    econom_ind_and_inflation_2010_2018,
    dev_ind_2010_2018_core,
    left_on=['country', 'year'],
    right_on=['country_name', 'year'],
    how='outer',
    indicator=True
)

merge_check['_merge'].value_counts()

In [ ]:
merge_check.loc[
    merge_check['_merge'] == 'left_only',
].sort_values(['country', 'year'], ascending=[True, False])

In [ ]:
merge_check.loc[
    merge_check['_merge'] == 'right_only',
].sort_values(['country_name', 'year'], ascending=[False, False])

In [ ]:
# Country mapping
country_name_mapping = {
    "USA": "United States",
    "UK": "United Kingdom",
    "Russia": "Russian Federation",
    "South Korea": "Korea, Rep."
}

econom_ind_and_inflation_2010_2018['country'] = (
    econom_ind_and_inflation_2010_2018['country'].replace(country_name_mapping)
)

merge_check = pd.merge(
    econom_ind_and_inflation_2010_2018,
    dev_ind_2010_2018_core,
    left_on=["country", "year"],
    right_on=["country_name", "year"],
    how="outer",
    indicator=True
)

print(merge_check["_merge"].value_counts())



After standardizing country names, the merge check showed 171 matched country-year records and 0 unmatched records from the Economic Indicators dataset. This means that all Economic Indicators records successfully matched with the World Bank dataset. The remaining 1728 right_only records belong only to the World Bank dataset, which contains many more countries.

In [ ]:
economic_worldbank_merged = pd.merge(
    econom_ind_and_inflation_2010_2018,
    dev_ind_2010_2018_core,
    left_on=["country", "year"],
    right_on=["country_name", "year"],
    how='inner',
)

economic_worldbank_merged.shape


In [ ]:
economic_worldbank_merged.info()

In [ ]:
# Fix typo

economic_worldbank_merged = economic_worldbank_merged.rename(
    columns={
        'gdb_usd_worldbank': 'gdp_usd_worldbank',
        'gdb_usd_economic': 'gdp_usd_economic',
    }
)

economic_worldbank_merged.info()

In [ ]:
economic_worldbank_merged


#### Analytic Features

In [ ]:
economic_worldbank_merged['gdp_difference'] = (
        economic_worldbank_merged['gdp_usd_economic'] -  economic_worldbank_merged['gdp_usd_worldbank']
)

economic_worldbank_merged['gdp_absolute_difference'] = (
    economic_worldbank_merged['gdp_difference'].abs()
)
economic_worldbank_merged[
    [
        "country",
        "year",
        "gdp_usd_economic",
        "gdp_usd_worldbank",
        "gdp_difference",
        "gdp_absolute_difference"
    ]
]



In [ ]:
economic_worldbank_merged['gdp_percentage_difference'] = (
    economic_worldbank_merged['gdp_absolute_difference'] /
    economic_worldbank_merged['gdp_usd_worldbank'] * 100
)

print(economic_worldbank_merged[
    [
        "country",
        "year",
        "gdp_usd_economic",
        "gdp_usd_worldbank",
        "gdp_difference",
        "gdp_absolute_difference",
        "gdp_percentage_difference"
    ]
].sort_values('gdp_percentage_difference', ascending=False))

In [ ]:
print(econom_ind_and_inflation_2010_2018.loc[
    econom_ind_and_inflation_2010_2018["country"].isin(["Indonesia", "Turkey"]),
    ["country", "year", "gdp_in_billion_usd", "gdb_usd_economic"]
].sort_values(["country", "year"]))

In [ ]:
# World Bank GDP as the main GDP variable for analysis
economic_worldbank_merged['gdp_usd_final'] = economic_worldbank_merged['gdp_usd_worldbank']

After converting GDP from the Economic Indicators dataset from billion USD to USD, GDP values from both datasets were compared. Most values were reasonably similar, but some observations showed extremely large percentage differences. In particular, several GDP values for Indonesia and Turkey in the Economic Indicators dataset were recorded as 1.0 or 2.0 billion USD, while the World Bank values showed GDP levels close to hundreds of billions or around one trillion USD. This suggests a possible scale or data-entry issue in the Economic Indicators dataset.

Because of this inconsistency, the World Bank GDP variable was used as the main GDP measure for further analysis. The original Economic Indicators GDP variable was kept for comparison purposes, and GDP difference columns were retained to assess consistency between the two sources.

In [ ]:
# Flag suspicious rows
economic_worldbank_merged['gdp_diff_status'] = np.where(
    economic_worldbank_merged['gdp_percentage_difference'] > 20,
    "Suspicious Difference",
    "Acceptable Difference"
)

economic_worldbank_merged[
    [
        "country",
        "year",
        "gdp_usd_economic",
        "gdp_usd_worldbank",
        "gdp_difference",
        "gdp_absolute_difference",
        "gdp_percentage_difference",
        'gdp_diff_status'
    ]
]


In [ ]:
suspicious_diffs = economic_worldbank_merged.loc[
    economic_worldbank_merged['gdp_diff_status'] == "Suspicious Difference",
]

print(f'Suspicious Differences Count: {len(suspicious_diffs)}')



In [ ]:
suspicious_diffs[
    [
        "country",
        "year",
        "gdp_usd_economic",
        "gdp_usd_worldbank",
        "gdp_difference",
        "gdp_absolute_difference",
        "gdp_percentage_difference",
        'gdp_diff_status'
    ]
]

Using a 20% threshold, 40 country-year observations were flagged as having suspicious GDP differences between the Economic Indicators dataset and the World Bank dataset. These observations were not removed automatically. Instead, they were flagged as potential scale or data-entry issues. For the final GDP-based analysis, the World Bank GDP variable was used as the main GDP measure because it was more consistent across countries and years.

In [ ]:
print(economic_worldbank_merged['gdp_per_capita_usd'].describe())

In [ ]:
# Set categories
q1, q3 = get_q1_q3(economic_worldbank_merged, 'gdp_per_capita_usd')

economic_worldbank_merged = add_categories(
    economic_worldbank_merged,
    'gdp_per_capita_usd',
    'gdp_per_capita_category',
    bins=[0, q1, q3, float("inf")],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True,
)

economic_worldbank_merged[['country', 'year','gdp_per_capita_usd' ,'gdp_per_capita_category']].sort_values('gdp_per_capita_usd', ascending=False)

The quantile method was used to create data-driven categories. The 25th percentile and 75th percentile were used as thresholds. This means that the lowest 25% of GDP per capita observations were classified as Low, the middle 50% as Medium, and the highest 25% as High.

In [ ]:
# Inflation Categories
zero_below = np.nextafter(0, -np.inf)

economic_worldbank_merged = add_categories(
    economic_worldbank_merged,
    'inflation_rate',
    'inflation_category',
    bins=[-float("inf"), zero_below, 3, 10,  float("inf")],
    labels=['Deflation','Low', 'Medium', 'High'],
    right=True
)

economic_worldbank_merged[
    [
        "country",
        "year",
        'inflation_rate',
        'inflation_category',
    ]
].sort_values('inflation_category', ascending=True)

Inflation categories were defined using economic interpretation. Values below 0% were classified as deflation. Inflation from 0% to 3% was classified as low inflation, inflation above 3% and up to 10% as moderate inflation, and inflation above 10% as high inflation. These thresholds were chosen because low positive inflation is generally considered stable, while inflation above 10% indicates high inflation pressure.

In [ ]:
economic_worldbank_merged['unemployment_rate'].describe()

In [ ]:
# Unemployment category
q1_unemployment, q3_unemployment = get_q1_q3(economic_worldbank_merged, 'unemployment_rate')

economic_worldbank_merged = add_categories(
    df=economic_worldbank_merged,
    col='unemployment_rate',
    category_col_name='unemployment_category',
    bins=[0, q1_unemployment, q3_unemployment, float("inf")],
    labels=[
        'Low unemployment',
        'Medium unemployment',
        'High unemployment',
    ],
    include_lowest=True
)


economic_worldbank_merged[
    [
        "country",
        "year",
        'unemployment_rate',
        'unemployment_category',
    ]
].sort_values('unemployment_rate', ascending=False)


Unemployment categories were created using the distribution of unemployment rates in the merged dataset. The 25th percentile and 75th percentile were used as thresholds. Observations below the 25th percentile were classified as Low unemployment, observations between the 25th and 75th percentile as Medium unemployment, and observations above the 75th percentile as High unemployment. This data-driven approach was chosen because unemployment levels differ across countries and the categories should reflect the distribution within this analytical sample.

In [ ]:
economic_worldbank_merged[['country', 'individuals_using_the_internet_of_population']]

In [ ]:
# Internet Usage categories
q1, q3 = get_q1_q3(economic_worldbank_merged, 'individuals_using_the_internet_of_population')

economic_worldbank_merged = add_categories(
    economic_worldbank_merged,
    col='individuals_using_the_internet_of_population',
    category_col_name='internet_usage_category',
    bins=[0, q1, q3, float("inf")],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

economic_worldbank_merged[['country', 'individuals_using_the_internet_of_population', 'internet_usage_category']]


In [ ]:
# Handle NaN values in Internet Usage Category
economic_worldbank_merged["internet_usage_category"] = (
    economic_worldbank_merged["internet_usage_category"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

economic_worldbank_merged[['country', 'individuals_using_the_internet_of_population', 'internet_usage_category']]

In [ ]:
#National Population Growth
economic_worldbank_merged['population_growth'] = (
    economic_worldbank_merged['birth_rate_crude_per_1_000_people'] -
    economic_worldbank_merged['death_rate_crude_per_1_000_people']
)

economic_worldbank_merged[
    [
        'country',
        'year',
        'birth_rate_crude_per_1_000_people',
        'death_rate_crude_per_1_000_people',
        'population_growth'
    ]
].sort_values('population_growth', ascending=False)

In [ ]:
# Average GDP, inflation, unemployment and economic growth per country
avg_economy_by_country = (
    economic_worldbank_merged
    .groupby('country')
    .agg(
        avg_gdp=('gdp_usd_final', 'mean'),
        avg_inflation=('inflation_rate', 'mean'),
        avg_unemployment=('unemployment_rate', 'mean'),
        avg_economic_growth=('economic_growth', 'mean'),
    )
    .reset_index()
)

avg_economy_by_country.sort_values('avg_inflation', ascending=True)

In [ ]:
# Average GDP, GDP per capita, unemployment, and life expectancy by region
avg_economy_by_region = (
    economic_worldbank_merged
    .groupby('region')
    .agg(
        avg_gdp=('gdp_usd_final', 'mean'),
        avg_gdp_per_capita=('gdp_per_capita_usd', 'mean'),
        avg_unemployment=('unemployment_rate', 'mean'),
        avg_life_expectancy=('life_expectancy_at_birth_years', 'mean'),
    )
    .reset_index()
)

avg_economy_by_region

In [ ]:
# Average inflation and unemployment by income group
avg_inflation_unemployment_by_income_group = (
    economic_worldbank_merged
    .groupby('income_group')
    .agg(
        avg_inflation=('inflation_rate', 'mean'),
        avg_unemployment=('unemployment_rate', 'mean'),
    )
    .reset_index()
)

avg_inflation_unemployment_by_income_group

In [ ]:
economic_worldbank_merged.inflation_category.unique()

In [ ]:
# Countries with the highest and lowest average inflation
highest_inflation_country = highest(avg_economy_by_country, 'avg_inflation')
lowest_inflation_country = lowest(avg_economy_by_country, 'avg_inflation')

print(highest_print('Inflation', highest_inflation_country['country'], highest_inflation_country['avg_inflation']))
print()
print(lowest_print('Inflation', lowest_inflation_country['country'], lowest_inflation_country['avg_inflation']))

In [ ]:
# Countries with the highest and lowest average unemployment
highest_avg_unemployment_country =highest(avg_economy_by_country, 'avg_unemployment')
lowest_avg_unemployment_country  = lowest(avg_economy_by_country, 'avg_unemployment')

print(highest_print('Unemployment', highest_avg_unemployment_country['country'], highest_avg_unemployment_country['avg_unemployment']))
print()
print(lowest_print('Unemployment', lowest_avg_unemployment_country['country'], lowest_avg_unemployment_country['avg_unemployment']))

In [ ]:
# Countries with the highest and lowest GDP per capita
highest_gdp_country= highest(avg_economy_by_country, 'avg_gdp')
lowest_gdp_country = lowest(avg_economy_by_country, 'avg_gdp')

print(highest_print('GDP', highest_gdp_country['country'], highest_gdp_country['avg_gdp']))
print()
print(lowest_print('GDP', lowest_gdp_country['country'], lowest_gdp_country['avg_gdp']))


In [ ]:
# Average internet usage by region and income group
avg_internet_usage_income_group = (
    economic_worldbank_merged
    .groupby('income_group')
    .agg(
        avg_internet_usage=('individuals_using_the_internet_of_population', 'mean'),
    )
    .reset_index()
)

avg_internet_usage_income_group

In [ ]:
# Relationship between income group and life expectancy.
life_expectancy_by_income_group = (
    economic_worldbank_merged
    .groupby('income_group')
    .agg(
        avg_life_expectancy=('life_expectancy_at_birth_years', 'mean'),
    )
    .reset_index()
)

life_expectancy_by_income_group

In [ ]:
# Average birth rate and death rate by region.
avg_birth_death_rate_by_region = (
    economic_worldbank_merged
    .groupby('region')
    .agg(
        avg_birth_rate=('birth_rate_crude_per_1_000_people', 'mean'),
        avg_death_rate=('death_rate_crude_per_1_000_people', 'mean'),
    )
    .reset_index()
)

avg_birth_death_rate_by_region

In [ ]:
# Countries with the strongest economic growth over time
economy_growth_top5 = avg_economy_by_country.sort_values('avg_economic_growth', ascending=False).head(5)[['country','avg_economic_growth']]
economy_growth_top5

In [ ]:
economic_worldbank_merged.info()

In [ ]:
# GDP difference between both tables
gdp_difference = economic_worldbank_merged[['country', 'year', 'gdp_difference', 'gdp_absolute_difference']]
gdp_difference['gdp_difference_percentage'] = (
        economic_worldbank_merged['gdp_absolute_difference'] /
        economic_worldbank_merged['gdp_usd_worldbank'] * 100
)
gdp_difference

In [ ]:
# Comparing Unemployment Rate from Economic Indicators And Inflation.csv with Unemployment from WorldBank.xlsx
economic_worldbank_merged['unemployment_difference'] = (
    economic_worldbank_merged['unemployment_rate'] -
    economic_worldbank_merged['unemployment_of_total_labor_force_modeled_ilo_estimate']
)

economic_worldbank_merged['unemployment_abs_difference'] = (
    economic_worldbank_merged['unemployment_difference'].abs()
)

unemployment_comparison =  economic_worldbank_merged[['country', 'year', 'unemployment_difference', 'unemployment_abs_difference']]
unemployment_comparison


In [ ]:
# Top 5 GDP difference between the two tables
top5_gdp_diff = (
    economic_worldbank_merged
    .groupby('country')
    .agg(avg_diff=('gdp_absolute_difference', 'mean'))
    .reset_index()
    .sort_values('avg_diff', ascending=False)
).head(5)

top5_gdp_diff



In [ ]:
# Assess whether unemployment values are similar across both sources
economic_worldbank_merged['unemployment_abs_difference'].describe()

The unemployment values from the two datasets appear to be mostly similar. The average absolute difference is approximately 1.05 percentage points, while the median difference is only 0.20 percentage points. This means that at least half of the country-year observations have very small differences between the two unemployment measures. However, the maximum difference is 9.41 percentage points, indicating that some observations differ substantially, possibly because of different data sources, estimation methods, or revisions.

In [ ]:
economic_worldbank_merged.info()

In [ ]:
# Difference by Region
difference_by_region  = (
    economic_worldbank_merged
    .groupby('region')
    .agg(
        avg_gdp_percentage_diff=('gdp_percentage_difference', 'mean'),
        avg_unemployment_abs_diff=('unemployment_abs_difference', 'mean'),
        observations_included=('country', 'count'),
    )
    .reset_index()
    .sort_values('avg_gdp_percentage_diff', ascending=False)
)

difference_by_region


The regional comparison shows that GDP differences between the two datasets are not equally distributed across regions. East Asia & Pacific has the highest average GDP percentage difference at around 24.82%, followed by Europe & Central Asia at around 23.41%. This suggests that GDP inconsistencies are more concentrated in these regions. This is likely influenced by the previously identified scale issues for countries such as Indonesia and Turkey.

For unemployment, the largest average absolute difference appears in Middle East & North Africa, with an average difference of about 4.81 percentage points. South Asia also shows a relatively higher unemployment difference at about 2.66 percentage points. In contrast, North America has the smallest unemployment difference at about 0.12 percentage points, suggesting that unemployment values are very similar between the two datasets for this region.

Overall, the results suggest that there are some systematic differences by region. GDP differences are largest in East Asia & Pacific and Europe & Central Asia, while unemployment differences are largest in Middle East & North Africa and South Asia.

In [ ]:
# GDP growth trend by country
gdp_growth_trend = first_and_last_year(
    economic_worldbank_merged, 'country', 'gdp_usd_final', ['country', 'year'], 'first_gdp', 'last_gdp'
)

gdp_growth_trend = calculate_change(gdp_growth_trend, 'first_gdp', 'last_gdp', 'gdp', percent=False)

gdp_growth_trend.sort_values('gdp_change_percent', ascending=False)

In [ ]:
# GDP Inflation trend by country
inflation_trend = first_and_last_year(
    economic_worldbank_merged,
    'country',
    'inflation_rate',
    ['country', 'year'],
    'first_inflation_rate',
    'last_inflation_rate'
)


inflation_trend = calculate_change(inflation_trend, 'first_inflation_rate', 'last_inflation_rate', 'inflation')

inflation_trend.sort_values('inflation_change_percent', ascending=False)


In [ ]:
# Unemployment trend by country
unemployment_trend = first_and_last_year(
    economic_worldbank_merged,
    'country',
    'unemployment_rate',
    ['country', 'year'],
    'first_unemployment_rate',
    'last_unemployment_rate'
)

unemployment_trend = calculate_change(
    unemployment_trend,
    'first_unemployment_rate',
    'last_unemployment_rate',
    'unemployment'
)

unemployment_trend.sort_values('unemployment_change_percent', ascending=False)

In [ ]:
# Internet usage over time

internet_usage_trend = first_and_last_year(
    economic_worldbank_merged,
    'country',
    'individuals_using_the_internet_of_population',
    ['country', 'year'],
    'first_internet_usage',
    'last_internet_usage'
)

internet_usage_trend = calculate_change(
    internet_usage_trend,
    'first_internet_usage',
    'last_internet_usage',
    'internet_usage',
    percent=False
)

internet_usage_trend.sort_values('internet_usage_change_percent', ascending=False)

In [ ]:
# Life expectancy change
life_expectancy_trend = first_and_last_year(
    economic_worldbank_merged,
    'country',
    'life_expectancy_at_birth_years',
    ['country', 'year'],
    'first_life_expectancy',
    'last_life_expectancy'
)

life_expectancy_trend = calculate_change(
    life_expectancy_trend,
    'first_life_expectancy',
    'last_life_expectancy',
    'life_expectancy',
    percent=False
)

life_expectancy_trend.sort_values('life_expectancy_change_percent', ascending=False)

In [ ]:
# GDP per capita trend
gpd_per_capita_trend = first_and_last_year(
    economic_worldbank_merged,
    'country',
    'gdp_per_capita_usd',
    ['country', 'year'],
    'first_gdp_per_capita_usd',
    'last_gdp_per_capita_usd'
)

gdp_per_capita_trend = calculate_change(
    gpd_per_capita_trend,
    'first_gdp_per_capita_usd',
    'last_gdp_per_capita_usd',
    'gdp_per_capita_usd',
    percent=False

)

gdp_per_capita_trend.sort_values('gdp_per_capita_usd_change_percent', ascending=False)


In [ ]:
# Top 3 GDP growth
top3_gdp_growth = top3(gdp_growth_trend, 'gdp_change_percent', ascending=False)
top3_gdp_growth

In [ ]:
# Highest increase in Internet usage
top3_internet_usage_increase = top3(internet_usage_trend, 'internet_usage_change_percent', ascending=False)
top3_internet_usage_increase

In [ ]:
# Highest Unemployment increase
top3_unemployment_increase = top3(unemployment_trend, 'unemployment_change_percent', ascending=False)
top3_unemployment_increase

In [ ]:
# Life expectancy improvement top 3
top3_life_expectancy_improvement = top3(life_expectancy_trend, 'life_expectancy_change_percent', ascending=False)
top3_life_expectancy_improvement

#### Country Development Profile

In [ ]:
# Country Development profile tabel

country_profile  = (
    economic_worldbank_merged
    .groupby(['country', 'country_code', 'region', 'income_group'])
    .agg(
        first_available_year=('year', 'min'),
        last_available_year=('year', 'max'),
        avg_gdp=('gdp_usd_final', 'mean'),
        avg_gdp_per_capita=('gdp_per_capita_usd', 'mean'),
        avg_inflation_rate=('inflation_rate', 'mean'),
        avg_unemployment_rate=('unemployment_rate', 'mean'),
        avg_economic_growth=('economic_growth', 'mean'),
        avg_life_expectancy=('life_expectancy_at_birth_years', 'mean'),
        avg_internet_usage=('individuals_using_the_internet_of_population', 'mean'),
        avg_birth_rate=('birth_rate_crude_per_1_000_people', 'mean'),
        avg_death_rate=('death_rate_crude_per_1_000_people', 'mean'),
        avg_population_denisity=('population_density_people_per_sq_km_of_land_area', 'mean'),
    )
    .reset_index()
)

country_profile

In [ ]:
# Development Status
gdp_per_capita_median = country_profile['avg_gdp_per_capita'].median()
life_expectancy_median = country_profile['avg_life_expectancy'].median()
internet_usage_median = country_profile['avg_internet_usage'].median()

country_profile['development_score'] = (
    (country_profile['avg_gdp_per_capita'] >= gdp_per_capita_median).astype(int) +
    (country_profile['avg_life_expectancy'] >= life_expectancy_median).astype(int) +
    (country_profile['avg_internet_usage'] >= internet_usage_median).astype(int)
)

development_status_map = {
    0: "Low development",
    1: "Medium development",
    2: "Medium development",
    3: "High development"
}

country_profile['development_status'] = (
    country_profile['development_score'].map(development_status_map)
)

country_profile




#### Visualisations

In [ ]:
# Average inflation rate by country
x = country_profile['country']
y = country_profile['avg_inflation_rate']

plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.title('Average Inflation Rate by Country')
plt.ylabel('AVG Inflation rate %')
plt.xlabel('Country')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# GDP per capita by income group
gdp_per_capita_by_income_group = (
    economic_worldbank_merged
    .groupby('income_group')
    .agg(
        avg_gdp_per_capita=('gdp_per_capita_usd', 'mean'),
    )
    .reset_index()
)

plt.figure(figsize=(10, 6))
plt.barh(
    gdp_per_capita_by_income_group['income_group'],
    gdp_per_capita_by_income_group['avg_gdp_per_capita'],
)
plt.title(' GDP by Income Group')
plt.ylabel('GDP')
plt.xlabel('Income Group')
plt.xticks()
plt.tight_layout()
plt.show()





In [ ]:
# Internet usage over time by region

internet_usage_by_region = (
    economic_worldbank_merged
    .groupby(['region', 'year'])
    .agg(
        avg_internet_usage=('individuals_using_the_internet_of_population', 'mean'),
    )
    .reset_index()
)

internet_usage_by_region

In [ ]:
# Internet usage over time by region plot
plt.figure(figsize=(10, 6))

for region in internet_usage_by_region["region"].unique():
    region_data = internet_usage_by_region[
        internet_usage_by_region["region"] == region
    ]

    plt.plot(
        region_data["year"],
        region_data["avg_internet_usage"],
        marker="o",
        label=region
    )

plt.title("Internet Usage Over Time by Region")
plt.xlabel("Year")
plt.ylabel("Average Internet Usage (% of Population)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Life expectancy vs. GDP per capita
plt.figure(figsize=(10, 6))

plt.scatter(
    country_profile["avg_gdp_per_capita"],
    country_profile["avg_life_expectancy"]
)

plt.title("Life Expectancy vs. GDP per Capita")
plt.xlabel("Average GDP per Capita (USD)")
plt.ylabel("Average Life Expectancy at Birth (Years)")
plt.tight_layout()
plt.show()

#### Exports

In [ ]:
# Merged Country-Year dataset
economic_worldbank_merged.to_csv('../outputs/task03/economic_data_merged.csv', index=False)

In [ ]:
# Country Development Profile
country_profile.to_excel('../outputs/task03/country_profile.xlsx', index=False)

In [ ]:
# Regional Summary
difference_by_region.to_excel('../outputs/task03/difference_by_region.xlsx', index=False)